# Задание 3. Задание для претендущих на оценку 8 и более баллов (выполняется по желанию).
Перед началом работы загрузите из личного кабинете файлы, на которых можно проверить код с «грязными» данными: transactions_dirty.csv и transactions_dirty.json.

Реализуйте для классов аккаунтов CheckingAccount (расчётный счёт) и SavingsAccount (сберегательный счёт) два метода:

● Метод загрузки истории в аккаунт из файла с транзакциями (файл транзакций общий для всех аккаунтов, необходимо учесть фильтрацию загружаемых значений).

● Метод clean_history(), который ищет ошибки в данных перед записью транзакций в историю (опечатки, отрицательные суммы, неверные даты). Обратите внимание, что для SavingsAccount (сберегательный счёт) доступно три типа операции (deposit, withdraw и interest), в то время как для CheckingAccount (расчётный счёт) доступны только два типа операции (deposit, withdraw). Все данные с ошибками считаем невалидными и не записываем в историю операций.

● После загрузки истории операций в аккаунт, баланс счёта должен обновиться.

# ==================Комментарий=================
Для избежания дубликации кода, метод clean_history был реализован в родительском классе Account. Так же в родительском классе был реализован метод для парсинга файлов parse_file.

В каждом наследнике было добавлено свойство account_type_code, в котором хранится тип аккаунта (checking или savings) в родителе реализован связывающий публичный метод load_account_history, в котором используются методы clean_history и parse_file.
# ==============================================

In [76]:
from datetime import datetime
from pathlib import Path
from typing import Any
import json
import csv
import re


class Account:
    accounts_number_order: int = 1
    account_type_code = "base"

    def __init__(self, account_holder: str, balance: float = 0):
        self.account_holder = None

        self.set_account_holder(account_holder)

        self.account_number: str = "ACC-1" + str(Account.accounts_number_order).zfill(5)
        self.__operations_history: dict = {}
        self.__account_counter = 1000
        self._balance: float = 0

        self.deposit(balance)

        Account.accounts_number_order += 1

    def get_balance(self) -> float:
        return self._balance

    def get_operations_history(self) -> str:
        """
        Получение истории операций

        :return: str
        """
        result = ''

        for date, operations in self.__operations_history.items():
            if len(operations) == 0:
                continue

            result += f"{date}:\n"

            for operation in operations:
                result += "   " + " | ".join(f"{k}: {v}" for k, v in operation.items()) + "\n"

        return result

    def deposit(self, amount: float) -> bool:
        """
        Внесение суммы на счет

        :param amount:
        :raises ValueError:
        :return: True
        """
        status = amount >= 0

        if status:
            self._balance += amount

        self.log_new_operation('deposit', amount, status)

        if not status:
            raise ValueError('Сумма должна быть не отрицательной!')

        return status

    def withdraw(self, amount: float) -> bool:
        """
        Снятие денег со счета

        :param amount:
        :raises ValueError:
        :return:
        """
        error: str | False = False

        if amount < 0:
            error = 'Сумма должна быть положительной!'

        elif amount > self.get_balance():
            error = 'Сумма должна быть меньше текущего баланса!'

        else:
            self._balance -= amount

        self.log_new_operation('withdraw', amount, not bool(error))

        if error:
            raise ValueError(error)

        return True

    def log_new_operation(self, operation_type: str, data: Any, status: bool) -> None:
        """
        Сохранение новой операции

        :param operation_type:
        :param data:
        :param status:
        :return:
        """
        current_date = datetime.now()
        date = current_date.strftime("%Y-%m-%d")
        time = current_date.strftime("%H:%M:%S")

        self.__operations_history.setdefault(date, []).append({
            'time': time,
            'operation_type': operation_type,
            'status': 'success' if status else 'fail',
            'balance': self.get_balance(),
            'data': data,
        })

    def set_account_holder(self, name: str) -> None:
        """
        Валидация и присваивание значения в поле account_holder

        :param name:
        :raises ValueError:
        :return:
        """
        parts = name.split()

        if len(parts) != 2 or " " * 2 in parts:
            raise ValueError("Имя владельца должно быть в формате: 'Имя Фамилия'")

        test = name.replace('-', '')

        if not test.istitle():
            raise ValueError("Имя и фамилия должны быть с большой буквы, и только они")

        if not test.replace(' ', '').isalpha():
            raise ValueError("Цифры и спец. символы (кроме дефиса) не допустимы")

        if re.search(r'[а-яёА-ЯЁ]', test) and re.search(r'[a-zA-Z]', test):
            raise ValueError("Смешивание кириллицы и латиницы не допустимо")

        self.account_holder = name

    def get_last_large_operations(self, count: int, by_date: str | None = None) -> str:
        """
        Получение последних крупных операций

        :param count: Выводимое кол-во операций
        :param by_date: Опционально - по дате
        :return:
        """
        result = ''

        i = count

        for date, operations in self.__operations_history.items():
            if i <= 0:
                break

            if by_date and by_date not in date:
                continue

            large_date_operations = ''

            for operation in operations:
                if i <= 0:
                    break

                if not isinstance(operation['data'], int | float) or operation['data'] < 1000:
                    continue

                large_date_operations += "\n   " + " | ".join(f"{k}: {v}" for k, v in operation.items())
                i -= 1

            if len(large_date_operations) > 0:
                result += f"{date}:{large_date_operations}"

        return result

    @staticmethod
    def _parse_file(file_path: str) -> list:
        """
        Парсинг локально-загружаемого файла. Поддерживаемые типы: JSON, CSV

        :param file_path:
        :raises FileNotFoundError:
        :raises ValueError:
        :return:
        """
        path = Path(file_path)

        if not path.is_file():
            raise FileNotFoundError(f"Файл по пути {file_path} не найден.")

        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                if path.suffix == '.json':
                    parsed_data = list(json.load(file))

                elif path.suffix == '.csv':
                    parsed_data = list(csv.DictReader(file))

                else:
                    raise ValueError("Неподдерживаемое расширение файла")

        except (json.JSONDecodeError, csv.Error) as e:
            raise ValueError(f"Ошибка чтения файла, {e}")

        return parsed_data

    def _clean_history(self, data: list) -> None:
        """
        Валидация с отчисткой и загрузкой истории транзакции

        :param data:
        :return:
        """
        operations_name_map = {'interest': 'apply_interest'}
        valid_statuses = {'success', 'fail'}

        def is_float(value: Any) -> bool:
            try:
                return float(value) > 0

            except (TypeError, ValueError):
                return False

        def is_correct_date(value: Any) -> bool:
            try:
                return bool(datetime.strptime(value, "%Y-%m-%d %H:%M:%S"))

            except (TypeError, ValueError):
                return False

        for row in data:
            if not (
                    row['account_type'] == self.account_type_code and
                    row['account_number'] == self.account_number and
                    row['status'] in valid_statuses and
                    is_float(row['amount']) and
                    is_float(row['balance_after']) and
                    is_correct_date(row['date']) and
                    isinstance(row['operation'], str) and
                    hasattr(self, operations_name_map.get(row['operation'], str(row['operation'])))
            ):
                continue

            date = row['date'].split()

            self.__operations_history.setdefault(date[0], []).append({
                'time': date[1],
                'operation_type': row['operation'],
                'status': row['status'],
                'balance': float(row['balance_after']),
                'data': float(row['amount'])
            })

        for operations in self.__operations_history.values():
            operations.sort(key=lambda x: x.get('time'))

        self.__operations_history = dict(sorted(self.__operations_history.items()))
        self._balance = list(self.__operations_history.values())[-1][-1]['balance']

    def load_account_history(self, file_path: str) -> None:
        """
        Загрузка истории аккаунта из файла. Поддерживаемые типы: JSON, CSV

        :param file_path:
        :raises ValueError:
        :return:
        """
        data = self._parse_file(file_path)

        self._clean_history(data)


class CheckingAccount(Account):
    account_type = "Расчетный счет"
    account_type_code = "checking"


class SavingsAccount(Account):
    account_type = "Сберегательный счет"
    account_type_code = "savings"

    def apply_interest(self, rate: int) -> float:
        """
        Расчет и применение процентов на остаток

        :param rate:
        :return:
        """
        interest = self.get_balance() * (rate / 100)

        self._balance -= interest

        self.log_new_operation('apply_interest', interest, True)

        return interest

    def withdraw(self, amount: float) -> bool:
        """
        Переопределение метода снятия со счета. Не позволяет снять больше 50% от текущей суммы

        :param amount:
        :raises ValueError:
        :return:
        """
        error: str | False = False

        if amount < 0:
            error = 'Сумма должна быть положительной!'

        elif amount > self.get_balance():
            error = 'Сумма должна быть меньше текущего баланса!'

        elif amount > self.get_balance() // 2:
            error = 'Сумма не должна быть больше 50% текущего баланса!'

        else:
            self._balance -= amount

        self.log_new_operation('withdraw', amount, not bool(error))

        if error:
            raise ValueError(error)

        return True


c = CheckingAccount('Тест Тестов', 1500)
s = SavingsAccount('Тест Тестов', 1000)

c.withdraw(500)

s.apply_interest(5)
s.load_account_history('transactions_dirty.csv')

print(s.get_operations_history())


15
2025-09-27:
   time: 22:17:26 | operation_type: deposit | status: success | balance: 1969.0 | data: 469.0
   time: 22:17:26 | operation_type: withdraw | status: success | balance: 1577.0 | data: 392.0
2025-09-28:
   time: 22:17:26 | operation_type: deposit | status: success | balance: 1894.0 | data: 317.0
2025-10-02:
   time: 22:17:26 | operation_type: deposit | status: success | balance: 3665.55 | data: 840.0
2025-10-06:
   time: 22:17:26 | operation_type: interest | status: success | balance: 4041.27 | data: 192.44
2025-10-08:
   time: 22:17:26 | operation_type: deposit | status: success | balance: 4603.27 | data: 426.0
2025-10-09:
   time: 22:17:26 | operation_type: withdraw | status: success | balance: 5071.27 | data: 170.0
   time: 22:17:26 | operation_type: deposit | status: success | balance: 6125.27 | data: 622.0
2025-10-10:
   time: 22:17:26 | operation_type: interest | status: success | balance: 6431.53 | data: 306.26
2025-10-12:
   time: 22:17:26 | operation_type: withdra